In [4]:
import os
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain_groq import ChatGroq
from langchain.tools import tool

c:\Users\Admin\Documents\GEN_AI\LLM_Evaluation\Test_AI\Dev\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
### LLM Configuration that will be used within agent

load_dotenv()
llm = ChatGroq(model_name="llama-3.1-8b-instant")

In [ ]:
### Verify LLM is working

llm.invoke("What is ML? Explain in 20 words")

AIMessage(content='Machine Learning (ML) is a field using algorithms to analyze data, learn patterns, and make predictions or decisions automatically.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 25, 'prompt_tokens': 44, 'total_tokens': 69, 'completion_time': 0.041868368, 'prompt_time': 0.001982316, 'queue_time': 0.055141334, 'total_time': 0.043850684}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_7b3cfae3af', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--665b6224-6b31-4fdb-bbb2-3df4e2c6d103-0', usage_metadata={'input_tokens': 44, 'output_tokens': 25, 'total_tokens': 69})

In [7]:
### Create a tool that the agent can use


@tool
def add_numbers(a: int, b: int) -> int:
    '''Adds two numbers and returns the result.'''
    return a + b

In [8]:
### Create a tool that the agent can use

@tool
def substract_numbers(a: int, b: int) -> int:
    '''Substracts two numbers and returns the result.'''
    return a - b

In [23]:
from langchain_community.tools import DuckDuckGoSearchRun

search_tool = DuckDuckGoSearchRun()

In [24]:
### Tools list

tools = [add_numbers,substract_numbers,search_tool]

In [27]:
### Create the agent with the LLM and tools

agent = create_agent(llm, tools = tools)

In [11]:
response = agent.invoke({"messages": [{"role": "user", "content": "What is 2 plus 2?"}]})

In [12]:
response

{'messages': [HumanMessage(content='What is 2 plus 2?', additional_kwargs={}, response_metadata={}, id='c370a27c-95b9-4270-9874-b62342550dca'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '2mcw8k4mt', 'function': {'arguments': '{"a":2,"b":2}', 'name': 'add_numbers'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 19, 'prompt_tokens': 297, 'total_tokens': 316, 'completion_time': 0.021766597, 'prompt_time': 0.016241611, 'queue_time': 0.049869569, 'total_time': 0.038008208}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_ab04adca7d', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--32da1e41-70d5-43fb-a436-d349e360b6d0-0', tool_calls=[{'name': 'add_numbers', 'args': {'a': 2, 'b': 2}, 'id': '2mcw8k4mt', 'type': 'tool_call'}], usage_metadata={'input_tokens': 297, 'output_tokens': 19, 'total_tokens': 316}),
  ToolMessage(content='4', name='add_numbers', id=

In [13]:
### Fetch the user question from response object
response['messages'][0].content

'What is 2 plus 2?'

In [14]:
### Fetch the Agent answer from response object
response['messages'][-1].content

'The result of 2 plus 2 is 4.'

In [ ]:
### We are storing the tool that was called in a variable for further use


tool = "add_numbers"


### Latter when time permits I will show how to parse the tool calls from the response object

### Deep Eval Configuration

In [ ]:
### If you want to unset the local model configuration for Deepeval then run the below command

# !deepeval unset-local-model

🙌 OpenAI will still be used by default because OPENAI_API_KEY is set.


In [16]:
### Login to Confident AI 

import deepeval
deepeval.login(os.getenv("DEEPEVAL_API_KEY"))

🎉🥳 Congratulations! You've successfully logged in! 🙌

In [ ]:
### if you want to use Groq API as you LLM provider for Deepeval then please perform the below configuation

import os
os.environ["OPENAI_BASE_URL"] = ""  # Groq’s OpenAI-compatible API
os.environ["OPENAI_API_KEY"] = ""               # use your Groq key here

In [ ]:
### Set Groq model as local model in Deepeval

!deepeval set-local-model --model-name="openai/gpt-oss-20b" --base-url="" --api-key=""

Settings updated for this session. To persist, use --save=dotenv[:path] 
(default .env.local) or set DEEPEVAL_DEFAULT_SAVE=dotenv:.env.local
🙌 Congratulations! You're now using a local model `openai/gpt-oss-20b` for all
evals that require an LLM.


In [17]:
### Create a test case


from deepeval.test_case import LLMTestCase
from deepeval.metrics import ToolCorrectnessMetric
from deepeval.test_case import ToolCall

test_case = LLMTestCase(
    input=response['messages'][0].content,             ### User question
    tools_called=[ToolCall(name=tool)],                ### Tool called by the agent
    actual_output=response['messages'][-1].content,    ### Agent response
    expected_output=response['messages'][-1].content,  ### Ground truth answer 
    expected_tools=[ToolCall(name = 'add_numbers')] ###Expected tool to be called Ground truth
    )   



In [18]:
### Create an evaluation dataset
import deepeval
from deepeval.dataset import EvaluationDataset

In [19]:
### Add the test case to the evaluation dataset
dataset = EvaluationDataset()
dataset.add_test_case(test_case)

In [20]:
dataset

EvaluationDataset(test_cases=[LLMTestCase(input='What is 2 plus 2?', actual_output='The result of 2 plus 2 is 4.', expected_output='The result of 2 plus 2 is 4.', context=None, retrieval_context=None, additional_metadata=None, tools_called=[ToolCall(
    name="add_numbers"
)], comments=None, expected_tools=[ToolCall(
    name="add_numbers"
)], token_cost=None, completion_time=None, name=None, tags=None, mcp_servers=None, mcp_tools_called=None, mcp_resources_called=None, mcp_prompts_called=None)], goldens=[], _alias=None, _id=None, _multi_turn=False)

In [21]:
### We want to evaluate the tool correctness metric on this test case

from deepeval.metrics import ToolCorrectnessMetric

In [22]:
deepeval.evaluate(dataset.test_cases, metrics= [ToolCorrectnessMetric()])

✨ You're running DeepEval's latest Tool Correctness Metric! (using None, strict=False, async_mode=True)...

c:\Users\Admin\Documents\GEN_AI\LLM_Evaluation\Test_AI\Dev\.venv\Lib\site-packages\rich\live.py:256: UserWarning: 
install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')



Metrics Summary

  - ✅ Tool Correctness (score: 1.0, threshold: 0.5, strict: False, evaluation model: None, reason: [
	 Tool Calling Reason: All expected tools ['add_numbers'] were called (order not considered).
	 Tool Selection Reason: No available tools were provided to assess tool selection criteria
]
, error: None)

For test case:

  - input: What is 2 plus 2?
  - actual output: The result of 2 plus 2 is 4.
  - expected output: The result of 2 plus 2 is 4.
  - context: None
  - retrieval context: None


Overall Metric Pass Rates

Tool Correctness: 100.00% pass rate




⚠ WARNING: No hyperparameters logged.
» ]8;id=839667;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Done 🎉! View results on 
]8;id=724205;https://app.confident-ai.com/project/cmhdvo6st07abnu0gdsqju0yp/test-runs/cmhi8r9xu04pgmn0gvh477z3m/regression-testing\https://app.confident-ai.com/project/cmhdvo6st07abnu0gdsqju0yp/test-runs/cmhi8r9xu04pgmn0gvh477z3m/regression-testi]8;;\
]8;id=724205;https://app.confident-ai.com/project/cmhdvo6st07abnu0gdsqju0yp/test-runs/cmhi8r9xu04pgmn0gvh477z3m/regression-testing\ng]8;;\

EvaluationResult(test_results=[TestResult(name='test_case_0', success=True, metrics_data=[MetricData(name='Tool Correctness', threshold=0.5, success=True, score=1.0, reason="[\n\t Tool Calling Reason: All expected tools ['add_numbers'] were called (order not considered).\n\t Tool Selection Reason: No available tools were provided to assess tool selection criteria\n]\n", strict_mode=False, evaluation_model=None, error=None, evaluation_cost=0.0, verbose_logs='Expected Tools:\n[\n    ToolCall(\n        name="add_numbers"\n    )\n] \n \nTools Called:\n[\n    ToolCall(\n        name="add_numbers"\n    )\n] \n \nAvailable Tools: [] \n \nTool Selection Score: 1.0 \n \nTool Selection Reason: No available tools were provided to assess tool selection criteria')], conversational=False, multimodal=False, input='What is 2 plus 2?', actual_output='The result of 2 plus 2 is 4.', expected_output='The result of 2 plus 2 is 4.', context=None, retrieval_context=None, turns=None, additional_metadata=None)

In [ ]:
### Another test case to evaluate search tool

response = agent.invoke({"messages": [{"role": "user", "content": "Who won ICC Womens Cricket World Cup in 2025?"}]})

In [ ]:
### Create another test case for search tool usage

tool = "DuckDuckGoSearchRun"        ### Simulating and storing the tool name
# tool = "duckduckgo_search"        ### Actual tool name to be used in expected tools


test_case_2 = LLMTestCase(
    input=response['messages'][0].content,             ### User question
    tools_called=[ToolCall(name=tool)],                ### Tool called by the agent
    actual_output=response['messages'][-1].content,    ### Agent response
    expected_output=response['messages'][-1].content,  ### Ground truth answer 
    expected_tools=[ToolCall(name = 'duckduckgo_search')] ###Expected tool to be called Ground truth
    ) 

In [ ]:
### Here I am directly evaluating the test case without adding to dataset

deepeval.evaluate([test_case_2], metrics= [ToolCorrectnessMetric()])

✨ You're running DeepEval's latest Tool Correctness Metric! (using None, strict=False, async_mode=True)...

c:\Users\Admin\Documents\GEN_AI\LLM_Evaluation\Test_AI\Dev\.venv\Lib\site-packages\rich\live.py:256: UserWarning: 
install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')



Metrics Summary

  - ❌ Tool Correctness (score: 0.0, threshold: 0.5, strict: False, evaluation model: None, reason: [
	 Tool Calling Reason: Incomplete tool usage: missing tools [ToolCall(
    name="duckduckgo_search"
)]; expected ['duckduckgo_search'], called ['DuckDuckGoSearchRun']. See more details above.
	 Tool Selection Reason: No available tools were provided to assess tool selection criteria
]
, error: None)

For test case:

  - input: Who won ICC Womens Cricket World Cup in 2025?
  - actual output: India won the ICC Women's Cricket World Cup 2025.
  - expected output: India won the ICC Women's Cricket World Cup 2025.
  - context: None
  - retrieval context: None


Overall Metric Pass Rates

Tool Correctness: 0.00% pass rate




⚠ WARNING: No hyperparameters logged.
» ]8;id=212094;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Done 🎉! View results on 
]8;id=356016;https://app.confident-ai.com/project/cmhdvo6st07abnu0gdsqju0yp/test-runs/cmhia4k3c04uqmn0gog8wjtti/regression-testing\https://app.confident-ai.com/project/cmhdvo6st07abnu0gdsqju0yp/test-runs/cmhia4k3c04uqmn0gog8wjtti/regression-testi]8;;\
]8;id=356016;https://app.confident-ai.com/project/cmhdvo6st07abnu0gdsqju0yp/test-runs/cmhia4k3c04uqmn0gog8wjtti/regression-testing\ng]8;;\

EvaluationResult(test_results=[TestResult(name='test_case_0', success=False, metrics_data=[MetricData(name='Tool Correctness', threshold=0.5, success=False, score=0.0, reason='[\n\t Tool Calling Reason: Incomplete tool usage: missing tools [ToolCall(\n    name="duckduckgo_search"\n)]; expected [\'duckduckgo_search\'], called [\'DuckDuckGoSearchRun\']. See more details above.\n\t Tool Selection Reason: No available tools were provided to assess tool selection criteria\n]\n', strict_mode=False, evaluation_model=None, error=None, evaluation_cost=0.0, verbose_logs='Expected Tools:\n[\n    ToolCall(\n        name="duckduckgo_search"\n    )\n] \n \nTools Called:\n[\n    ToolCall(\n        name="DuckDuckGoSearchRun"\n    )\n] \n \nAvailable Tools: [] \n \nTool Selection Score: 1.0 \n \nTool Selection Reason: No available tools were provided to assess tool selection criteria')], conversational=False, multimodal=False, input='Who won ICC Womens Cricket World Cup in 2025?', actual_output="India 